# Título: Análisis descriptivo con segmentación temporal

### Installs

In [ ]:
# %pip install seaborn

### Imports

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("data_cleaning_2026_20_04.csv")

In [ ]:
df.info()

Transformación y reducción de datos

In [ ]:
df_clean = df.loc[:, ~df.columns.isin(['Pet', 'Weight', 'Height'])]


In [ ]:
df_clean['Month_absence'].unique()

In [ ]:
df_clean['Month_absence'] = df_clean['Month_absence'].astype('Int64')
df_clean.info()

In [ ]:
df_clean[df_clean['Absenteeism_hours'] == 0]

In [ ]:
df_clean[df_clean['Month_absence'].isna()]

Decidimos quitar los valores faltante por el tipo de datos, al no representar un ausencia real por el tipo de dataset.    

In [ ]:
df_clean = df_clean[df_clean['Absenteeism_hours'] > 0]
df_clean.info()

In [ ]:
df_clean.to_csv("data_transformation_2026_22_04.csv", index=False, encoding="utf-8")

### Magnitud  del ausentísmo

In [ ]:
df_clean['Absenteeism_hours'].describe()

In [ ]:
# Histograma de distribución general

plt.figure(figsize=(8,5))
sns.histplot(df_clean['Absenteeism_hours'], bins=20, kde=True)
plt.title('Distribución general de horas de absentismo')
plt.xlabel('Horas de absentismo')
plt.ylabel('Frecuencia')
plt.show()

Media inflada por casos extremos (120 h) - mean 7.6h y median 4h
Gran dispersion por presencia de casos extremos

Magnitud moderada en  promedio 7.6h pero altamente desigual: mediana de 4h y 75% con menos de 8h
Outliers incrementan la media y explica alta variabilidad.
Los percentiles son más adecuados para describir el comportamiento típico del absentísmo y no la media 

### Visualizar patrones temporales

Identificar meses criticos, estacionalidad y variabilidad

In [ ]:
# Agrupación por mes

df_month = df_clean.groupby('Month_absence')['Absenteeism_hours'].agg(
    total_absence='sum',
    mean_absence='mean',
    median_absence='median',
    count='count'
).reset_index()

df_month


In [ ]:
# Gráfico de barras: total de horas por mes (meses criticos)

plt.figure(figsize=(10,5))
sns.barplot(data=df_month,x='Month_absence', y='total_absence')
plt.title('Total de horas de absentismo por mes')
plt.xlabel('Mes')
plt.ylabel('Horas totales')
plt.show()

In [ ]:
# boxplot por mes para ver dispersion y outliers (variabilidad)

plt.figure(figsize=(12,5))
sns.boxplot(data=df_clean, x='Month_absence', y='Absenteeism_hours')
plt.title('Distribución del absentismo por mes')
plt.xlabel('Mes')
plt.ylabel('Horas de absentismo')
plt.show()

### Variación estacional

In [ ]:
# Grafico por estacion - (estacionalidad)

# Agrupación por estación
df_season = df_clean.groupby('Seasons')['Absenteeism_hours'].agg(
    total_absence='sum',
    mean_absence='mean',
    median_absence='median',
    count='count'
).reset_index()

df_season

In [ ]:
# Gráfico comparativo por estación

plt.figure(figsize=(8,5))
sns.barplot(data=df_season, x='Seasons', y='total_absence')
plt.title('Total de horas de absentismo por estación')
plt.xlabel('Estación')
plt.ylabel('Horas totales')
plt.show()

Más riesgo de ausencias prolongadas en invierno, verano y primavera con niveles similares 

### Perfiles de riesgo asociados al tiempo

In [ ]:
# Comparación de absentismo por estación y presencia de enfermedad

plt.figure(figsize=(10,5))
sns.boxplot(data=df, x='Seasons', y='Absenteeism_hours', hue='Disease_flag')
plt.title('Absentismo por estación y condición de salud')
plt.xlabel('Estación')
plt.ylabel('Horas de absentismo')
plt.show()

In [ ]:
# Cruzamos estación y tipo de ausencia
 
# total de horas por estación y motivo

df_cross = df_clean.groupby(['Seasons', 'Reason_absence_mapped'])['Absenteeism_hours'].sum().reset_index()
df_cross.sort_values(['Seasons', 'Absenteeism_hours'], ascending=[True, False])

In [ ]:
# visualización

plt.figure(figsize=(14,6))
sns.barplot(
    data=df_cross,
    x='Seasons',
    y='Absenteeism_hours',
    hue='Reason_absence_mapped',
    palette='tab20'
)
plt.title('Horas de absentismo por estación y motivo')
plt.xlabel('Estación')
plt.ylabel('Horas totales')
plt.legend(title='Motivo de ausencia', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
# motivos por estación

df_cross.groupby('Seasons').apply(
    lambda x: x.sort_values('Absenteeism_hours', ascending=False).head(3)
)

### Identificación meses críticos

In [ ]:
# Top 3 meses con mayor absentismo total

df_month.sort_values('total_absence', ascending=False).head(3)

In [ ]:
# Top 3 meses con mayor absentismo promedio

df_month.sort_values('mean_absence', ascending=False).head(3)

### Conclusiones

In [ ]:
print("Mes con mayor absentismo total:", 
      df_month.loc[df_month['total_absence'].idxmax(), 'Month_absence'])

print("Estación con mayor absentismo total:", 
      df_season.loc[df_season['total_absence'].idxmax(), 'Seasons'])